# Notebook 01 — Collecte météo
**Pipeline :** Top 35 villes FR → coordonnées GPS (Nominatim) → prévisions 7 jours (OpenWeatherMap) → score météo → `cities.csv`

## 0. Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import plotly.express as px
from pathlib import Path
from dotenv import load_dotenv

# Rendre src/ importable depuis notebooks/
sys.path.append(str(Path().resolve().parent))

from src.nominatim import get_all_coordinates
from src.weather import collect_weather_data, compute_weather_score

load_dotenv(dotenv_path=Path().resolve().parent / ".env")
OWM_API_KEY = os.getenv("OWM_API_KEY")

assert OWM_API_KEY, "OWM_API_KEY manquante dans le fichier .env !"
print("Clé OWM : OK")

## 1. Liste des 35 villes

In [ ]:
CITIES = [
"Mont Saint Michel","St Malo","Bayeux","Le Havre","Rouen","Paris",
"Amiens","Lille","Strasbourg","Chateau du Haut Koenigsbourg","Colmar",
"Eguisheim","Besancon","Dijon","Annecy","Grenoble","Lyon",
"Gorges du Verdon","Bormes les Mimosas","Cassis","Marseille",
"Aix en Provence","Avignon","Uzes","Nimes","Aigues Mortes",
"Saintes Maries de la mer","Collioure","Carcassonne","Ariege",
"Toulouse","Montauban","Biarritz","Bayonne","La Rochelle"
]
print(f"{len(CITIES)} villes")

## 2. Coordonnées GPS via Nominatim

In [ ]:
# Cellule 2 — Coordonnées GPS
from src.nominatim import get_all_coordinates

df_cities = get_all_coordinates(CITIES)
print(f"\nShape : {df_cities.shape}")
df_cities.head()

## 3. Prévisions météo via OpenWeatherMap

In [ ]:
df_weather = collect_weather_data(df_cities, OWM_API_KEY, delay=0.5)

print(f"Shape : {df_weather.shape}  ({len(df_cities)} villes × 7 jours)")
df_weather.head()

## 4. Score météo par ville

from sklearn.preprocessing import MinMaxScaler

# Normaliser chaque feature entre 0 et 1, puis pondérer
score = (temp_norm × 0.5) - (pop_norm × 0.3) - (rain_norm × 0.2)  
coeff. modifiables dans `src/weather.py`.

In [ ]:
df_score = compute_weather_score(df_weather)

print("Top 10 destinations :")
df_score[["rank", "city", "temp_moy", "pop_moy", "rain_total", "weather_score"]].head(10)

## 5. Carte Plotly — Top 5 destinations

In [ ]:
top5 = df_score.head(5)

# Normaliser le score pour l'utiliser comme taille et ajouter +2 pour éviter 0
score_min = df_score["weather_score"].min()
df_score["size_plot"] = df_score["weather_score"] - score_min + 2  

fig = px.scatter_mapbox(
    df_score,
    lat="lat", lon="lon",
    hover_name="city",
    hover_data={"weather_score": True, "temp_moy": ":.1f", "rain_total": ":.1f", "size_plot": False},
    color="weather_score",
    size="size_plot",                  
    color_continuous_scale="RdYlGn",
    size_max=20,
    zoom=4.5,
    center={"lat": 46.5, "lon": 2.5},
    mapbox_style="carto-positron",
    title="Top 35 villes françaises — score météo 7 jours"
)

for _, r in top5.iterrows():
    fig.add_annotation(
        x=r["lon"], y=r["lat"],
        text=f"#{int(r['rank'])} {r['city']}",
        showarrow=False,
        font=dict(size=11, color="darkgreen"),
        bgcolor="white",
        opacity=0.85
    )

fig.show()

## 6. Sauvegarde

In [ ]:
out = Path().resolve().parent / "data" / "raw" / "cities.csv"
df_score.to_csv(out, index=False)
print(f"Sauvegardé : {out}")
print(f"Colonnes   : {list(df_score.columns)}")
print(f"Lignes     : {len(df_score)}")